# EHB 466E — 2021 Manavgat & Marmaris yanmış alan haritalama
## Faz 1: Veri hattı + dNBR benchmark + EFFIS doğrulaması (ara sunum demosu)

Bu notebook baştan sona **tek seferde** çalışır (Runtime → Run all):
1. EFFIS poligonundan çalışma alanını (AOI) otomatik çıkarır
2. Planetary Computer STAC'tan Sentinel-2 L2A sahnelerini arar; AOI birden fazla tile'a düşüyorsa aynı geçişin tile'larını **mozaikler**, SCL ile AOI içindeki bulutsuzluk oranına bakarak en iyi tarihi seçer
3. Sadece AOI'yi okur (tüm tile değil) → birkaç yüz MB RAM
4. SCL bulut/gölge/su maskesi uygular, NBR / dNBR hesaplar
5. 3 eşik (USGS 0.10, USGS 0.27, Otsu) ile ikili harita üretir
6. EFFIS ile Precision / Recall / F1 / IoU / alan hesaplar; şekilleri ve tabloyu `outputs/` klasörüne kaydeder

**Colab'de çalıştırma:** aşağıdaki ilk hücre repoyu klonlar; EFFIS verisi repoda `data/effis_TR_2021.json` olarak durur, elle yükleme gerekmez.

In [ ]:
# Colab'de: repoyu klonla ve içine geç (kendi bilgisayarınızda repo klasöründen açtıysanız bu hücre atlanır)
import os
REPO_URL = "https://github.com/html37/ehb466-burnt-area-tr.git"
if "google.colab" in str(get_ipython()) and not os.path.exists("data/effis_TR_2021.json"):
    repo = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(repo):
        !git clone -q {REPO_URL}
    os.chdir(repo)
print("Çalışma klasörü:", os.getcwd())

In [ ]:
# Colab'de eksik paketleri kur (geopandas/matplotlib Colab'de zaten var)
!pip -q install pystac-client planetary-computer rioxarray

In [ ]:
import os, time, resource, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
import pystac_client, planetary_computer
from shapely.geometry import shape, box
from shapely.ops import unary_union
from rioxarray.merge import merge_arrays
from rasterio import features
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- AYARLAR ----------------
EFFIS_PATH = next((p for p in ["data/effis_TR_2021.json", "effis_TR_2021.json"] if os.path.exists(p)), None)
assert EFFIS_PATH, "EFFIS dosyası bulunamadı: data/effis_TR_2021.json"
OUT_DIR = "outputs"
BUFFER_M = 3000          # AOI = EFFIS poligonu + 3 km tampon
MAX_TILE_CLOUD = 20      # STAC aramasında tile düzeyinde bulut üst sınırı (%)
MIN_CLEAR = 0.97         # AOI içinde istenen bulutsuz piksel oranı

EVENTS = {
    # effis_id: EFFIS dosyasındaki poligon kimliği
    # pre: yangından önceki arama penceresi, post: yangın bittikten sonraki pencere
    "Manavgat": dict(effis_id=51931, pre=("2021-07-01", "2021-07-27"), post=("2021-08-07", "2021-09-15")),
    "Marmaris": dict(effis_id=51942, pre=("2021-07-01", "2021-07-28"), post=("2021-08-06", "2021-09-15")),
}
THRESHOLDS = {"dNBR>0.10 (USGS low)": 0.10, "dNBR>0.27 (USGS moderate)": 0.27, "Otsu": "otsu"}

# SCL sınıfları: 0 nodata, 1 saturated, 2 dark area, 3 cloud shadow, 4 vegetation, 5 bare,
# 6 water, 7 unclassified, 8/9 cloud, 10 cirrus, 11 snow
# Not: yanık izleri sık sık "2 dark area" olarak sınıflanır, bu yüzden 2'yi MASKELEMİYORUZ.
SCL_VALID_LAND = [2, 4, 5, 7, 11]
SCL_CLEAR = SCL_VALID_LAND + [6]

os.makedirs(OUT_DIR, exist_ok=True)
T0 = time.perf_counter()

### 1. EFFIS referans verisi ve çalışma alanları

In [ ]:
import json
with open(EFFIS_PATH, encoding="utf-8") as f:
    gj = json.load(f)
crs = gj.get("crs", {}).get("properties", {}).get("name", "EPSG:3035").replace("urn:ogc:def:crs:", "").replace("::", ":")
effis = gpd.GeoDataFrame.from_features(gj["features"], crs=crs)   # EPSG:3035 (ETRS89-LAEA)
for col in ["initialdate", "finaldate"]:
    effis[col] = pd.to_datetime(effis[col].astype(str).str[:10].str.replace("/", "-"))
assert "id" in effis.columns, f"'id' sütunu yok, mevcut sütunlar: {list(effis.columns)}"

for name, cfg in EVENTS.items():
    poly = effis[effis["id"] == cfg["effis_id"]]
    assert len(poly) == 1, f"{name}: EFFIS id {cfg['effis_id']} bulunamadı"
    cfg["bbox"] = [float(v) for v in poly.buffer(BUFFER_M).to_crs(4326).total_bounds]
    r = poly.iloc[0]
    print(f"{name:9s} EFFIS id={r['id']}  {r['initialdate'].date()} → {r['finaldate'].date()}  "
          f"{r['area_ha']:>6} ha   AOI bbox={np.round(cfg['bbox'], 3).tolist()}")

### 2. Sentinel-2 sahne seçimi (aynı geçişin tile mozaiği + AOI içi bulut kontrolü)

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

def search(bbox, dates):
    s = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox,
                       datetime=f"{dates[0]}/{dates[1]}",
                       query={"eo:cloud_cover": {"lt": MAX_TILE_CLOUD}})
    return list(s.items())

def coverage(item, aoi_box):
    return shape(item.geometry).intersection(aoi_box).area / aoi_box.area

def read_asset(item, asset, bbox):
    # Sadece AOI penceresini okur (COG), tüm tile'ı belleğe almaz
    da = rioxarray.open_rasterio(item.assets[asset].href).squeeze("band", drop=True)
    return da.rio.clip_box(*bbox, crs="EPSG:4326").rio.write_nodata(0)

def read_group(items, asset, bbox, aoi_box):
    # Aynı gün/aynı geçişe ait tile'ları okuyup tek bir mozaikte birleştirir
    parts = []
    for it in sorted(items, key=lambda i: -coverage(i, aoi_box)):
        try:
            da = read_asset(it, asset, bbox).load()
        except Exception:          # tile AOI ile kesişmiyorsa
            continue
        if parts and da.rio.crs != parts[0].rio.crs:   # farklı UTM dilimi ise ilkine yeniden projekte et
            da = da.rio.reproject(parts[0].rio.crs, resolution=abs(parts[0].rio.resolution()[0]),
                                  nodata=0, resampling=Resampling.nearest)
        parts.append(da)
    return parts[0] if len(parts) == 1 else merge_arrays(parts, nodata=0)

def group_by_pass(items, aoi_box):
    # Aynı gün + aynı uydu = aynı geçiş; birlikte AOI'yi ne kadar kapladıklarını hesapla
    groups = {}
    for it in items:
        groups.setdefault((it.datetime.date(), it.properties.get("platform", "")), []).append(it)
    out = []
    for (day, plat), its in groups.items():
        cov = unary_union([shape(i.geometry) for i in its]).intersection(aoi_box).area / aoi_box.area
        out.append(dict(day=day, platform=plat, items=its, coverage=cov))
    return out

def pick_scene(items, bbox, aoi_box, latest_first):
    # En yakın tarihten başlayarak AOI'yi kaplayan ve bulutsuz olan ilk geçişi seç
    cands = sorted(group_by_pass(items, aoi_box), key=lambda g: g["day"], reverse=latest_first)[:6]
    best = None
    for g in cands:
        if g["coverage"] < 0.5:
            continue
        scl = read_group(g["items"], "SCL", bbox, aoi_box)
        g["scl"] = scl
        # Bulutsuz piksel sayısı / AOI'nin tamamındaki beklenen piksel sayısı (kapsanmayan kısım da cezalandırılır)
        x0, y0, x1, y1 = transform_bounds("EPSG:4326", scl.rio.crs, *bbox)
        rx, ry = scl.rio.resolution()
        g["clear"] = min(float(np.isin(scl.values, SCL_CLEAR).sum()) / ((x1 - x0) * (y1 - y0) / abs(rx * ry)), 1.0)
        if best is None or g["clear"] > best["clear"]:
            best = g
        if g["clear"] >= MIN_CLEAR:
            break
    assert best is not None, "Uygun sahne bulunamadı"
    return best

scene_rows = []
for name, cfg in EVENTS.items():
    bbox = cfg["bbox"]; aoi_box = box(*bbox)
    pre = pick_scene(search(bbox, cfg["pre"]), bbox, aoi_box, latest_first=True)
    post = pick_scene(search(bbox, cfg["post"]), bbox, aoi_box, latest_first=False)
    cfg["pre_group"], cfg["post_group"] = pre, post
    cfg["pre_item"], cfg["post_item"] = pre["items"][0], post["items"][0]     # tarih bilgisi için
    for role, g in [("öncesi", pre), ("sonrası", post)]:
        tiles = sorted(i.properties["s2:mgrs_tile"] for i in g["items"])
        print(f"{name} {role}: {g['day']} {g['platform']} | tile(lar) {', '.join(tiles)} | "
              f"AOI kaplama %{100*g['coverage']:.1f} | AOI bulutsuz %{100*g['clear']:.1f}")
        if g["coverage"] < 0.98:
            print("  ⚠ AOI'nin bir kısmı bu geçişte görüntülenmemiş")
        for it in g["items"]:
            scene_rows.append({"event": name, "role": "pre" if role == "öncesi" else "post",
                               "date": str(g["day"]), "item_id": it.id,
                               "mgrs_tile": it.properties["s2:mgrs_tile"],
                               "tile_cloud_pct": it.properties["eo:cloud_cover"],
                               "aoi_coverage_pct": round(100 * g["coverage"], 2),
                               "aoi_clear_pct": round(100 * g["clear"], 2)})
pd.DataFrame(scene_rows).to_csv(f"{OUT_DIR}/selected_scenes.csv", index=False)   # tekrarlanabilirlik kaydı

### 3. Yansıtım, NBR ve dNBR

In [ ]:
def boa_offset(item):
    # 25 Ocak 2022 sonrası (processing baseline >= 04.00) L2A verilerinde -1000 ofset var
    try:
        return -1000.0 if float(item.properties.get("s2:processing_baseline", "0")) >= 4.0 else 0.0
    except ValueError:
        return 0.0

def reflectance(group, asset, bbox, ref=None):
    aoi_box = box(*bbox)
    raw = read_group(group["items"], asset, bbox, aoi_box)
    da = ((raw.astype("float32") + boa_offset(group["items"][0])) / 10000.0).where(raw != 0)
    da = da.rio.write_nodata(np.nan)
    if ref is not None and (da.shape != ref.shape or da.rio.transform() != ref.rio.transform()):
        da = da.rio.reproject_match(ref, resampling=Resampling.average)   # ör. 10 m B04 → 20 m ızgara
    return da

def nbr(nir, swir):
    return (nir - swir) / (nir + swir)

for name, cfg in EVENTS.items():
    t = time.perf_counter()
    bbox, pre, post = cfg["bbox"], cfg["pre_group"], cfg["post_group"]
    ref = reflectance(pre, "B12", bbox)                 # 20 m referans ızgara (mozaik)
    pre_b8a  = reflectance(pre,  "B8A", bbox, ref)
    post_b8a = reflectance(post, "B8A", bbox, ref)
    post_b12 = reflectance(post, "B12", bbox, ref)
    post_b04 = reflectance(post, "B04", bbox, ref)      # sadece görselleştirme için
    pre_scl  = pre["scl"].rio.reproject_match(ref, resampling=Resampling.nearest).values
    post_scl = post["scl"].rio.reproject_match(ref, resampling=Resampling.nearest).values

    dnbr = nbr(pre_b8a, ref) - nbr(post_b8a, post_b12)
    valid = (np.isin(pre_scl, SCL_VALID_LAND) & np.isin(post_scl, SCL_VALID_LAND)
             & np.isfinite(dnbr.values))
    cfg.update(ref=ref, dnbr=dnbr, valid=valid,
               rgb=np.dstack([post_b12.values, post_b8a.values, post_b04.values]))
    dnbr.where(valid).rio.to_raster(f"{OUT_DIR}/{name}_dnbr.tif")    # ara sonuç önbelleği
    print(f"{name}: ızgara {dnbr.shape} @ {abs(ref.rio.resolution()[0]):.0f} m, "
          f"geçerli kara pikseli %{100*valid.mean():.1f}, süre {time.perf_counter()-t:.1f} s")

### 4. EFFIS referans maskesi, eşikleme ve metrikler

In [ ]:
def effis_mask(ref, pre_date, post_date):
    # Öncesi ve sonrası görüntü tarihleri arasında başlayan tüm EFFIS yangınları
    sel = effis[(effis["initialdate"] >= pre_date) & (effis["initialdate"] <= post_date)].to_crs(ref.rio.crs)
    sel = sel[sel.intersects(box(*ref.rio.bounds()))]
    m = features.rasterize(((g, 1) for g in sel.geometry), out_shape=ref.shape,
                           transform=ref.rio.transform(), fill=0, dtype="uint8")
    return m.astype(bool), sel

def otsu(values, bins=512):
    v = np.clip(values[np.isfinite(values)], -0.5, 1.3)
    hist, edges = np.histogram(v, bins=bins)
    c = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist); w1 = w0[-1] - w0
    m = np.cumsum(hist * c)
    mu0 = m / np.maximum(w0, 1); mu1 = (m[-1] - m) / np.maximum(w1, 1)
    return float(c[np.argmax(w0 * w1 * (mu0 - mu1) ** 2)])

def metrics(pred, ref, valid, px_ha):
    p, r = pred[valid], ref[valid]
    tp = int(np.sum(p & r)); fp = int(np.sum(p & ~r)); fn = int(np.sum(~p & r)); tn = int(np.sum(~p & ~r))
    return {
        "Precision": tp / (tp + fp) if tp + fp else 0.0,
        "Recall": tp / (tp + fn) if tp + fn else 0.0,
        "F1": 2 * tp / (2 * tp + fp + fn) if tp else 0.0,
        "IoU": tp / (tp + fp + fn) if tp else 0.0,
        "OA": (tp + tn) / max(tp + tn + fp + fn, 1),
        "Tahmin alanı (ha)": (tp + fp) * px_ha,
        "EFFIS alanı (ha)": (tp + fn) * px_ha,
    }

rows = []
for name, cfg in EVENTS.items():
    ref, dnbr, valid = cfg["ref"], cfg["dnbr"].values, cfg["valid"]
    pre_d = pd.Timestamp(cfg["pre_item"].datetime.date()); post_d = pd.Timestamp(cfg["post_item"].datetime.date())
    cfg["effis_ref"], cfg["effis_sel"] = effis_mask(ref, pre_d, post_d)
    px_ha = abs(ref.rio.resolution()[0] * ref.rio.resolution()[1]) / 1e4
    gt = cfg["effis_ref"]
    masked_pct = 100 * np.sum(gt & ~valid) / max(np.sum(gt), 1)   # EFFIS alanının maskede kalan kısmı
    cfg["preds"] = {}
    for label, thr in THRESHOLDS.items():
        thr_val = otsu(dnbr[valid]) if thr == "otsu" else thr
        pred = (dnbr > thr_val) & valid
        cfg["preds"][label] = (thr_val, pred)
        rows.append({"Olay": name, "Yöntem": label, "Eşik": round(thr_val, 3),
                     **metrics(pred, gt, valid, px_ha), "Maskelenen EFFIS (%)": masked_pct})

results = pd.DataFrame(rows)
results.to_csv(f"{OUT_DIR}/benchmark_dnbr_metrics.csv", index=False)
display(results.round({"Precision": 3, "Recall": 3, "F1": 3, "IoU": 3, "OA": 3,
                       "Tahmin alanı (ha)": 0, "EFFIS alanı (ha)": 0, "Maskelenen EFFIS (%)": 1}))

### 5. Şekiller (her olay için: genel bakış + 3 eşiğin hata haritası)

In [ ]:
from matplotlib.patches import Patch

def stretch(rgb):
    out = np.empty_like(rgb)
    for k in range(3):
        b = rgb[..., k]; lo, hi = np.nanpercentile(b, [2, 98])
        out[..., k] = np.clip((b - lo) / (hi - lo), 0, 1)
    return np.nan_to_num(out)

ERR_COLORS = ["#f2f2f2", "#d7301f", "#fdae61", "#2c7bb6", "#ffffff"]   # TN, TP, FP, FN, geçersiz
err_cmap = ListedColormap(ERR_COLORS)
legend = [Patch(color=ERR_COLORS[1], label="Doğru yanık (TP)"),
          Patch(color=ERR_COLORS[2], label="Yanlış alarm (FP)"),
          Patch(color=ERR_COLORS[3], label="Kaçırılan (FN)"),
          Patch(facecolor="white", edgecolor="grey", label="Maskelenmiş (bulut/su)")]

def clean(ax):
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(""); ax.set_ylabel("")

for name, cfg in EVENTS.items():
    ref, valid, gt = cfg["ref"], cfg["valid"], cfg["effis_ref"]
    l, b, r, t = ref.rio.bounds(); ext = (l, r, b, t)
    pre_d, post_d = cfg["pre_item"].datetime.date(), cfg["post_item"].datetime.date()

    # (a) Genel bakış: yanlış renkli görüntü + dNBR
    fig, ax = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    ax[0].imshow(stretch(cfg["rgb"]), extent=ext)
    cfg["effis_sel"].boundary.plot(ax=ax[0], color="yellow", lw=0.8)
    ax[0].set_title(f"Yangın sonrası {post_d} (SWIR2/NIR/Kırmızı) — sarı: EFFIS sınırı")
    im = ax[1].imshow(np.where(valid, cfg["dnbr"].values, np.nan), extent=ext,
                      cmap="RdYlGn_r", vmin=-0.2, vmax=0.9)
    fig.colorbar(im, ax=ax[1], shrink=0.8, label="dNBR")
    ax[1].set_title(f"dNBR ({pre_d} → {post_d})")
    for a in ax: clean(a); a.set_xlim(l, r); a.set_ylim(b, t)
    fig.suptitle(f"{name} 2021", fontsize=15)
    fig.savefig(f"{OUT_DIR}/{name}_overview.png", dpi=200); plt.show()

    # (b) Üç eşiğin hata haritaları — eşik seçimi EFFIS'e bakılarak YAPILMIYOR, hepsi gösteriliyor
    fig, ax = plt.subplots(1, len(cfg["preds"]), figsize=(6 * len(cfg["preds"]), 6), constrained_layout=True)
    for a, (label, (thr_val, pred)) in zip(np.atleast_1d(ax), cfg["preds"].items()):
        err = np.zeros(pred.shape, np.uint8)
        err[pred & gt] = 1; err[pred & ~gt] = 2; err[~pred & gt] = 3; err[~valid] = 4
        a.imshow(err, extent=ext, cmap=err_cmap, vmin=0, vmax=4, interpolation="nearest")
        m = results[(results.Olay == name) & (results.Yöntem == label)].iloc[0]
        a.set_title(f"{label}, eşik {thr_val:.2f}\nF1={m.F1:.3f}  IoU={m.IoU:.3f}  P={m.Precision:.3f}  R={m.Recall:.3f}")
        clean(a)
    fig.legend(handles=legend, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(f"{name} 2021 — dNBR benchmark, EFFIS'e göre hata haritaları", fontsize=15)
    fig.savefig(f"{OUT_DIR}/{name}_error_maps.png", dpi=200, bbox_inches="tight"); plt.show()

peak_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
print(f"Toplam süre: {(time.perf_counter() - T0)/60:.1f} dk | En yüksek RAM: {peak_mb:.0f} MB")
print(f"Çıktılar: {sorted(os.listdir(OUT_DIR))}")

### 6. Ortam bilgisi (tekrarlanabilirlik için)

In [ ]:
import sys, importlib.metadata as md_
print("Python", sys.version.split()[0])
for p in ["numpy", "pandas", "xarray", "rioxarray", "rasterio", "geopandas", "shapely", "pystac-client", "planetary-computer", "matplotlib"]:
    try: print(f"{p:20s} {md_.version(p)}")
    except md_.PackageNotFoundError: print(f"{p:20s} -")